In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window


def expand_calendar_to_hourly(raw_table: str):
    """
    Transform a raw daily calendar table into an hourly calendar table.

    The function reads a raw table from the bronze layer, where each row
    represents one calendar date, and expands each day into 24 hourly rows.
    It also derives time-based calendar features used later in training and
    inference datasets.

    Derived columns:
        - timestamp_utc: hourly timestamp for each hour of the day
        - hour_of_day: hour extracted from timestamp_utc
        - day_of_week: weekday number with Monday=1 and Sunday=7
        - is_weekend: 1 if the day is Saturday or Sunday, otherwise 0
        - is_holiday: 1 if the calendar description is not "weekend" or
          "working day", otherwise 0
        - month: month extracted from timestamp_utc
        - day_of_year: day number within the year

    Args:
        raw_table: Name of the raw bronze table to read from
            fortum_challenge_data.01_bronze.

    Returns:
        A Spark DataFrame where each original daily row has been expanded into
        hourly rows and enriched with derived calendar features.

    Side Effects:
        - Reads a table from the bronze layer using the global Spark session.
        - Prints progress information to standard output.
    """
    print(f"Exploding bronze table {raw_table}")
    df = spark.table(f"fortum_challenge_data.01_bronze.{raw_table}")

    df_hours = (
        df.withColumnRenamed("timestamp", "date")
        .withColumn(
            "timestamp_utc",
            F.explode(
                F.sequence(
                    F.col("date").cast("timestamp"),
                    F.col("date").cast("timestamp") + F.expr("INTERVAL 23 HOURS"),
                    F.expr("INTERVAL 1 HOUR")
                )
            )
        )
        .withColumn("hour_of_day", F.hour(F.col("timestamp_utc")))
        .withColumn(
        "day_of_week",
        F.expr("((dayofweek(timestamp_utc) + 5) % 7) + 1")
        )
        .withColumn(
            "is_weekend",
            F.when(F.col("day_of_week").isin(6,7), 1).otherwise(0)
        )
        .withColumn(
            "is_holiday",
            F.when(~F.col("desc").isin("weekend", "working day"), 1).otherwise(0)
        )
        .withColumn("month", F.month(F.col("timestamp_utc")))
        .withColumn("day_of_year", F.dayofyear(F.col("timestamp_utc")))
        .drop("date")
    )
    
    return df_hours


def split_hourly_train_inference(df):
    """
    Split hourly calendar data into training and inference datasets.

    The function orders the input DataFrame by timestamp, assigns a row number,
    removes the earliest rows to support lag-based feature engineering, and then
    splits the remaining data into:
        - training data before the inference window
        - inference data for the final 48-hour prediction window

    Current logic:
        - Drops the first 336 hourly rows
        - Uses 2024-09-29 00:00:00 as the start of the inference window
        - Uses 2024-10-01 00:00:00 as the end of the inference window

    Args:
        df: Hourly Spark DataFrame, typically created by expand_calendar_to_hourly().

    Returns:
        A tuple of two Spark DataFrames:
            - df_train: historical hourly training data
            - df_inference: final 48-hour hourly inference data

    Side Effects:
        Prints progress information to standard output.
    """
    print("Splitting a raw hourly table as training and inference tables")

    window = Window.orderBy("timestamp_utc")

    # Add row index
    df_indexed = df.withColumn(
        "row_num",
        F.row_number().over(window)
    )

    # Drop the first 336 rows to preserve enough history for lag features.
    df_cut = df_indexed.filter(F.col("row_num") > 336).drop("row_num")

    df_asc = df_cut.orderBy("timestamp_utc")

    cutoff_start = "2024-09-29 00:00:00"
    cutoff_end = "2024-10-01 00:00:00"


    # Historical/training part
    df_train = df_asc.filter(
        F.col("timestamp_utc") < F.lit(cutoff_start).cast("timestamp")
    )

    # Final 48h inference part
    df_inference = df_asc.filter(
        (F.col("timestamp_utc") >= F.lit(cutoff_start).cast("timestamp")) &
        (F.col("timestamp_utc") < F.lit(cutoff_end).cast("timestamp"))
    )
    print("Bundling hourly tables as a tuple")


    return df_train, df_inference


def prepare_monthly_train_inference(df_training, df_inference):
    """
    Prepare monthly training and inference calendar tables.

    The function orders the provided training and inference DataFrames by
    timestamp and returns them as a tuple. Unlike split_hourly_train_inference(), this
    function does not create a split itself; it assumes the input DataFrames
    already represent the intended monthly training and inference periods.

    Args:
        df_training: Spark DataFrame for the monthly training period.
        df_inference: Spark DataFrame for the monthly inference period.

    Returns:
        A tuple of two Spark DataFrames:
            - df_train_monthly: ordered monthly training data
            - df_inference_monthly: ordered monthly inference data

    Side Effects:
        Prints progress information to standard output.
    """
    print("Bundling monthly tables as a tuple")
    df_train_monthly = df_training.orderBy("timestamp_utc")
    df_inference_monthly = df_inference.orderBy("timestamp_utc")
    return df_train_monthly, df_inference_monthly


def write_to_silver_table(df, table_name: str):
    """
    Write a Spark DataFrame to a silver-layer table.

    The function overwrites the target table in the silver layer with the
    contents of the provided DataFrame.

    Args:
        df: Spark DataFrame to write.
        table_name: Name of the destination table under
            fortum_challenge_data.02_silver.

    Returns:
        None.

    Side Effects:
        - Overwrites any existing data in the destination table.
        - Writes to the silver layer in the metastore.
        - Prints progress information to standard output.
    """
    df.write \
        .mode("overwrite") \
        .saveAsTable(f"fortum_challenge_data.02_silver.{table_name}")
    print(f"Successfully wrote to silver table {table_name}")



if __name__ == "__main__":

    # Read and transform raw daily calendar tables into hourly calendar tables.
    raw_training_table = "calendar_data_raw"
    raw_monthly_inference_table = "calendar_monthly_inference_raw"

    training_table = expand_calendar_to_hourly(raw_training_table)
    monthly_inference_table = expand_calendar_to_hourly(raw_monthly_inference_table)

    # Split hourly data into training and final 48h inference sets.
    hourly_training_inference_table = split_hourly_train_inference(training_table)

    # Bundle monthly training and monthly inference tables.
    monthly_training_inference_table = prepare_monthly_train_inference(training_table, monthly_inference_table)


    clean_hourly_training_table = "calendar_hourly_training_clean"
    clean_hourly_inference_table = "calendar_hourly_inference_clean"
    clean_monthly_training_table = "calendar_monthly_training_clean"
    clean_monthly_inference_table = "calendar_monthly_inference_clean"

    write_to_silver_table(hourly_training_inference_table[0], clean_hourly_training_table)
    write_to_silver_table(hourly_training_inference_table[1], clean_hourly_inference_table)
    write_to_silver_table(monthly_training_inference_table[0], clean_monthly_training_table)
    write_to_silver_table(monthly_training_inference_table[1], clean_monthly_inference_table)

    monthly_training_inference_table[1].display()